# 02 - Text Preprocessing

## Objective

The objective is to clean the `designation` and `description` fields before modelling. The same fixed rules are applied to the labelled training catalogue and the official test catalogue.

Report 1 found missing descriptions, encoded HTML characters and HTML tags. The cleaning removes this formatting noise while preserving product names, accents, numbers, units and model codes. No feature extraction or model training is performed here.

## 1. Load the Data

The labelled training catalogue and official test catalogue are loaded separately. Each product keeps its original `row_id`.

In [1]:
from pathlib import Path

import pandas as pd

from src.data_utils import get_data_paths, load_merged_data
from src.text_utils import clean_text, combine_text

data_paths = get_data_paths()
labelled_raw = load_merged_data().rename_axis("row_id").reset_index()
official_test_raw = (
    pd.read_csv(data_paths["x_test"], index_col=0)
    .rename_axis("row_id")
    .reset_index()
)

assert labelled_raw["row_id"].is_unique
assert official_test_raw["row_id"].is_unique

pd.DataFrame({
    "catalogue": ["labelled training", "official test"],
    "products": [len(labelled_raw), len(official_test_raw)],
})

,catalogue,products
0,labelled training,84916
1,official test,13812


## 2. Clean the Text

For each catalogue, missing descriptions become empty text, HTML formatting is removed, uppercase is converted to lowercase, repeated spaces are removed, and the cleaned designation and description are combined.

In [2]:
def prepare_text(data):
    prepared = data.copy()
    prepared["description_available"] = prepared["description"].notna()
    prepared["designation_clean"] = prepared["designation"].fillna("").map(clean_text)
    prepared["description_clean"] = prepared["description"].fillna("").map(clean_text)
    prepared["text_combined"] = [
        combine_text(title, description)for title, description in zip(prepared["designation_clean"], prepared["description_clean"])
    ]
    return prepared


labelled_text = prepare_text(labelled_raw)
official_test_text = prepare_text(official_test_raw)
print("Both catalogues were cleaned using the same rules.")

Both catalogues were cleaned using the same rules.


## 3. Check Before and After Examples

The code automatically finds two short descriptions with HTML tags and two with encoded characters, then displays the original and cleaned versions.

In [3]:
descriptions = labelled_raw["description"].fillna("")
has_tag = descriptions.str.contains("<", regex=False)
has_entity = descriptions.str.contains("&", regex=False)
long_enough = descriptions.str.len() > 20

tag_ids = descriptions[has_tag & long_enough].str.len().nsmallest(2).index
entity_ids = descriptions[has_entity & ~has_tag & long_enough].str.len().nsmallest(2).index
example_ids = list(tag_ids) + list(entity_ids)

examples = pd.DataFrame({
    "before": labelled_raw.loc[example_ids, "description"],
    "after": labelled_text.loc[example_ids, "description_clean"],
})
pd.set_option("display.max_colwidth", None)
examples

,before,after
64477,<p>Volet de skimmer</p>,volet de skimmer
83375,<p>Raccord en T 1/4</p>,raccord en t 1/4
21098,pere noel mamas & papas,pere noel mamas & papas
48782,coffret de l&#39;écrivain,coffret de l'écrivain


## 4. Check Vocabulary Cleaning

Report 1 found HTML words such as `br`, `strong`, `ul` and `li` in the raw vocabulary. Their occurrences are counted before and after cleaning to verify that formatting terms are removed.

In [4]:
raw_text = (
    labelled_raw["designation"].fillna("") + " "
    + labelled_raw["description"].fillna("")
).str.lower()

noise_terms = ["br", "strong", "ul", "li"]
vocabulary_check = pd.DataFrame({
    "raw occurrences": [raw_text.str.count(rf"\b{term}\b").sum() for term in noise_terms],
    "cleaned occurrences": [labelled_text["text_combined"].str.count(rf"\b{term}\b").sum() for term in noise_terms],
}, index=noise_terms)

vocabulary_check

,raw occurrences,cleaned occurrences
br,124213,561
strong,50399,303
ul,14724,85
li,109355,1037


## 5. Check Repeated Text

Repeated cleaned titles are counted but not removed. Some may be legitimate catalogue entries, and some identical titles have different labels. This check uses only the labelled training catalogue.

In [5]:
title_counts = labelled_text["designation_clean"].value_counts()
repeated_titles = title_counts[title_counts > 1].index

label_counts = labelled_text.groupby("designation_clean")["prdtypecode"].nunique()
conflicting_titles = label_counts[label_counts > 1].index

pd.Series({
    "rows with repeated titles": labelled_text["designation_clean"].isin(repeated_titles).sum(),
    "repeated-title groups": len(repeated_titles),
    "groups with conflicting labels": len(conflicting_titles),
}).to_frame("count")

,count
rows with repeated titles,4120
repeated-title groups,1443
groups with conflicting labels,157


## 6. Attach the Shared Split

`02_preprocessing.ipynb` saves one training/validation assignment for every labelled `row_id`. Text and image preprocessing both join this same file, so they evaluate the same products. The official test catalogue remains separate.

In [6]:
split_path = Path(data_paths["x_train"]).parent.parent / "processed" / "data_split.csv"
split_manifest = pd.read_csv(split_path)
labelled_text = labelled_text.merge(
    split_manifest, on="row_id", how="left", validate="one_to_one"
)

assert labelled_text["split"].notna().all()
labelled_text["split"].value_counts().to_frame("products")

,products
split,
train,67932
validation,16984


## 7. Validate and Save

The checks confirm that no products were lost and every product has usable combined text. The original files remain unchanged.

In [7]:
assert len(labelled_text) == 84_916
assert len(official_test_text) == 13_812
assert labelled_text["text_combined"].str.len().gt(0).all()
assert official_test_text["text_combined"].str.len().gt(0).all()

common_columns = [
    "row_id", "designation_clean", "description_clean",
    "text_combined", "description_available",
]
labelled_output = labelled_text[common_columns + ["split", "prdtypecode"]]
official_test_output = official_test_text[common_columns]

output_dir = Path(data_paths["x_train"]).parent.parent / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
labelled_output.to_csv(output_dir / "text_train_processed.csv", index=False)
official_test_output.to_csv(output_dir / "text_test_processed.csv", index=False)

pd.DataFrame({
    "catalogue": ["labelled training", "official test"],
    "products_saved": [len(labelled_output), len(official_test_output)],
})

,catalogue,products_saved
0,labelled training,84916
1,official test,13812


## Conclusion

Both catalogues use the same fixed cleaning rules. The labelled text also carries the shared training/validation assignment used by the image workflow. Learned transformations such as TF-IDF must later be fitted on training rows only.